# 🎯 Tutorial: Matching Networks - Atención para Few-Shot

## Meta-Learning con Mecanismos de Atención

En este tutorial aprenderás:

- 🔍 Qué son Matching Networks (Vinyals et al., 2016)
- 🎯 Cómo usar atención para clasificación few-shot
- 💻 Implementación completa con episodic training
- 📊 Comparación con Prototypical Networks

---

## 📖 Parte 1: Teoría

### ¿Qué son Matching Networks?

**Matching Networks** (Vinyals et al., 2016) fue uno de los primeros trabajos en Meta-Learning moderno. La idea central:

> "La clasificación es un problema de matching: encuentra los ejemplos del support set más similares a tu query."

### Diferencias con Prototypical Networks:

| Aspecto | Prototypical Networks | Matching Networks |
|---------|----------------------|-------------------|
| Representación | **Prototipo** (promedio) | **Todos los ejemplos** |
| Comparación | Distancia euclidiana | **Atención ponderada** |
| Complejidad | O(N) | O(N*K) |
| Flexibilidad | Menos | Más (atención adaptativa) |

### Matemáticamente:

Para un query $\hat{x}$, la predicción es:

$$P(y | \hat{x}, S) = \sum_{i=1}^{|S|} a(\hat{x}, x_i) \cdot y_i$$

donde $a(\hat{x}, x_i)$ es la atención (similitud) entre el query y cada ejemplo de soporte:

$$a(\hat{x}, x_i) = \frac{\exp(\text{cosine}(f(\hat{x}), g(x_i)))}{\sum_j \exp(\text{cosine}(f(\hat{x}), g(x_j)))}$$

### Intuición:

```
Query: ?
Support Set:
  🐱 (cat) → Similitud: 0.8
  🐶 (dog) → Similitud: 0.1
  🐱 (cat) → Similitud: 0.7
  🐦 (bird) → Similitud: 0.05

Predicción = weighted vote:
  cat: 0.8 + 0.7 = 1.5  ← Ganador!
  dog: 0.1
  bird: 0.05
```


---

## 🛠️ Parte 2: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import print_success, print_hint, HintSystem, run_test
from utils.data_utils import create_classification_task, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup completo!")

---

## 💻 Parte 3: Ejercicio 1 - Mecanismo de Atención

El corazón de Matching Networks: calcular similitudes con atención.

**Tu tarea**: Implementa el mecanismo de atención.

In [ ]:
def cosine_similarity(x1, x2):
    """
    Calcula similitud coseno entre dos conjuntos de vectores.
    
    Args:
        x1: [N, D] - Primer conjunto de vectores
        x2: [M, D] - Segundo conjunto de vectores
    
    Returns:
        similarities: [N, M] - Matriz de similitudes coseno
    """
    # TODO: Implementa similitud coseno
    # Recuerda: cosine(a, b) = (a · b) / (||a|| * ||b||)
    # Pista: F.normalize() normaliza vectores, luego usa matmul
    
    pass  # TODO: Reemplaza con tu código


def compute_attention(query_embeddings, support_embeddings, temperature=1.0):
    """
    Calcula pesos de atención entre queries y support set.
    
    Args:
        query_embeddings: [n_query, D]
        support_embeddings: [n_support, D]
        temperature: Parámetro de temperatura para softmax
    
    Returns:
        attention: [n_query, n_support] - Pesos de atención
    """
    # TODO: Implementa el cálculo de atención
    # 1. Calcula similitudes coseno
    # 2. Divide por temperatura
    # 3. Aplica softmax sobre la dimensión de support
    
    pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_attention = HintSystem([
    "Para cosine_similarity: normaliza x1 y x2 con F.normalize(x, dim=1), luego x1 @ x2.T.",
    "Para compute_attention: usa cosine_similarity(query_embeddings, support_embeddings).",
    "Divide las similitudes por temperature, luego aplica F.softmax(similarities / temp, dim=1).",
    "dim=1 en softmax para que cada query tenga pesos que sumen 1 sobre todos los support examples."
])

In [ ]:
# Para ver pistas
hints_attention.show_hint()

In [ ]:
# ✅ TEST 1: Verificar similitud coseno

def test_cosine():
    # Vectores de prueba
    x1 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
    x2 = torch.tensor([[1.0, 0.0], [1.0, 1.0]])
    
    sim = cosine_similarity(x1, x2)
    
    assert sim.shape == (2, 2), f"Shape debe ser (2, 2), es {sim.shape}"
    
    # [1,0] y [1,0] deben tener similitud 1.0
    assert torch.allclose(sim[0, 0], torch.tensor(1.0), atol=1e-5), "Similitud [0,0] debe ser ~1.0"
    
    # [1,0] y [0,1] deben tener similitud 0.0
    assert torch.allclose(sim[0, 1], torch.tensor(0.707), atol=0.01), "Similitud [0,1] incorrecta"
    
    print_success("✅ Similitud coseno correcta!")

run_test(test_cosine, "Test de Similitud Coseno")

In [ ]:
# ✅ TEST 2: Verificar atención

def test_attention():
    query = torch.randn(3, 64)  # 3 queries
    support = torch.randn(10, 64)  # 10 support examples
    
    attn = compute_attention(query, support)
    
    assert attn.shape == (3, 10), f"Shape debe ser (3, 10), es {attn.shape}"
    
    # Cada query debe tener pesos que sumen 1
    sums = attn.sum(dim=1)
    assert torch.allclose(sums, torch.ones(3), atol=1e-5), "Atención debe sumar 1 para cada query"
    
    print_success("✅ Atención calculada correctamente!")

run_test(test_attention, "Test de Atención")

---

## 💻 Parte 4: Ejercicio 2 - Matching Networks Completa

**Tu tarea**: Implementa el método de predicción usando atención.

In [ ]:
class MatchingNetwork(nn.Module):
    """
    Matching Networks para Few-Shot Classification.
    """
    
    def __init__(self, input_channels=1, embedding_dim=64):
        super(MatchingNetwork, self).__init__()
        
        # Encoder (similar a Prototypical Networks)
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, embedding_dim, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
    
    def forward(self, support_x, support_y, query_x, n_way):
        """
        Forward pass usando atención.
        
        Args:
            support_x: [n_support, C, H, W]
            support_y: [n_support]
            query_x: [n_query, C, H, W]
            n_way: Número de clases
        
        Returns:
            logits: [n_query, n_way]
        """
        # TODO: Implementa el forward pass con atención
        # 1. Codifica support y query con self.encoder
        # 2. Calcula atención entre query y support
        # 3. Usa atención para hacer weighted vote sobre las labels
        # 4. Retorna logits por clase
        
        pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_matching = HintSystem([
    "Codifica: support_emb = self.encoder(support_x); query_emb = self.encoder(query_x).",
    "Calcula atención: attn = compute_attention(query_emb, support_emb).",
    "Convierte support_y a one-hot: support_one_hot = F.one_hot(support_y, n_way).float().",
    "Weighted vote: logits = torch.matmul(attn, support_one_hot). Esto da [n_query, n_way]."
])

In [ ]:
# Para ver pistas
hints_matching.show_hint()

In [ ]:
# ✅ TEST 3: Verificar Matching Network

def test_matching_network():
    model = MatchingNetwork(input_channels=1, embedding_dim=64)
    task = create_classification_task(n_way=5, k_shot=3, q_query=10)
    
    logits = model(
        task['x_support'],
        task['y_support'],
        task['x_query'],
        n_way=5
    )
    
    assert logits.shape == (50, 5), f"Logits debe ser (50, 5), es {logits.shape}"
    
    print_success("✅ Matching Network implementada correctamente!")

run_test(test_matching_network, "Test de Matching Network")

---

## 📊 Parte 5: Entrenamiento y Comparación

In [ ]:
def train_matching_net(model, n_episodes=500, n_way=5, k_shot=5, q_query=15, lr=0.001):
    """
    Entrena Matching Network.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    accuracies = []
    
    print(f"🚀 Entrenando Matching Networks por {n_episodes} episodios...\n")
    
    for episode in range(n_episodes):
        # Sample task
        task = create_classification_task(n_way=n_way, k_shot=k_shot, q_query=q_query)
        
        # Forward
        model.train()
        logits = model(
            task['x_support'].to(device),
            task['y_support'].to(device),
            task['x_query'].to(device),
            n_way=n_way
        )
        
        loss = criterion(logits, task['y_query'].to(device))
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        acc = (logits.argmax(dim=1) == task['y_query'].to(device)).float().mean().item()
        
        losses.append(loss.item())
        accuracies.append(acc)
        
        if (episode + 1) % 100 == 0:
            print(f"Episodio {episode+1}/{n_episodes} - Loss: {np.mean(losses[-100:]):.4f}, Acc: {np.mean(accuracies[-100:]):.4f}")
    
    return losses, accuracies


# Entrenar
matching_net = MatchingNetwork().to(device)
losses, accs = train_matching_net(matching_net, n_episodes=500)

print("\n✅ Entrenamiento completado!")

In [ ]:
# Visualizar curvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(losses, alpha=0.3)
ax1.plot(np.convolve(losses, np.ones(50)/50, mode='valid'), linewidth=2)
ax1.set_xlabel('Episodio')
ax1.set_ylabel('Loss')
ax1.set_title('Curva de Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(accs, alpha=0.3)
ax2.plot(np.convolve(accs, np.ones(50)/50, mode='valid'), linewidth=2)
ax2.set_xlabel('Episodio')
ax2.set_ylabel('Accuracy')
ax2.set_title('Curva de Accuracy')
ax2.axhline(y=0.2, color='r', linestyle='--', label='Random')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **Matching Networks** usan atención para comparar queries con support examples
2. A diferencia de prototipos, mantienen **todos los ejemplos** del support set
3. La predicción es un **weighted vote** basado en similitudes
4. Más flexible pero también más costoso que Prototypical Networks

### ⚖️ Prototypical vs Matching:

**Cuándo usar Prototypical:**
- ✅ Más eficiente (O(N) vs O(NK))
- ✅ Funciona bien cuando las clases son compactas
- ✅ Más simple de implementar y debuggear

**Cuándo usar Matching:**
- ✅ Clases con alta variabilidad intra-clase
- ✅ Cuando tienes suficiente K (5-shot o más)
- ✅ Necesitas interpretabilidad (puedes ver qué ejemplos contribuyen)

### 📊 Performance Típico:

En muchos benchmarks, ambos algoritmos tienen performance similar, con ligera ventaja para:
- **Prototypical**: En 1-shot y tareas simples
- **Matching**: En 5-shot+ y tareas complejas

### 🚀 Variantes y Mejoras:

- **Full Context Embeddings (FCE)**: Procesar todo el support set con atención bidireccional
- **Relation Networks**: Aprender la función de similitud en lugar de usar coseno
- **Matching + MAML**: Combinar ambos enfoques

---

## 🎉 ¡Ahora conoces ambos enfoques métricos principales!
